# Lab 15：在雲端架起自己的模型服務

教材：<https://tw-ytp.github.io/vocabuddy/#/lab15>　｜　附錄 F・特別篇（選修）

**開始前先確認**：右側 Settings 的 Accelerator 選 `GPU T4 x2`、Internet 設為 On。
兩者都需要先完成手機簡訊驗證（Settings → Phone verification）。

**三件事先知道**
1. 你的程式碼與截圖會離開你的電腦（經過 Cloudflare 與 Kaggle）。不要用在不能外流的專案。
2. 免費額度每週約 30 小時、單次 12 小時，**開著就計時**；用完要按 Stop Session，關分頁不會停。
3. 把 Notebook 當成對外 API 伺服器是否符合 Kaggle 使用條款，**教材沒有查證過**，請自行評估帳號風險。

版本與型號會過期，最新值見教材附錄 B：<https://tw-ytp.github.io/vocabuddy/#/fresh>

依序執行 Cell 1 → Cell 7，約十分鐘。**Cell 7 要最後才跑。**

## Cell 1：確認拿到兩張 GPU

應該看到**兩列** `Tesla T4`，各 15360 MiB。

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## Cell 2：安裝推理引擎

下載官方預編譯檔，不必自行編譯，約一分鐘。最後印出 `llama-server` 路徑就是成功。

In [ ]:
%%bash
set -e
TAG=b11435                      # 引擎版本：最新值見教材附錄 B
cd /kaggle/working && mkdir -p llama
for f in llama-$TAG-bin-ubuntu-cuda-12.8-x64.tar.gz cudart-llama-$TAG-bin-ubuntu-cuda-12.8-x64.tar.gz; do
  wget -q https://github.com/ggml-org/llama.cpp/releases/download/$TAG/$f
  tar -xzf $f -C llama && rm $f
done
find llama -name llama-server

## Cell 3：下載模型與視覺檔

約一到兩分鐘。出現 unauthenticated requests 的警告可以忽略，模型是公開的。

In [ ]:
from huggingface_hub import hf_hub_download

REPO   = "ConwayResearch/Underdog-Saluki-27B-1.0"          # 模型：最新值見教材附錄 B
MAIN   = "Underdog-Saluki-27B-1.0-IQ2-mix.gguf"            # 主模型，約 7.89 GB
VISION = "mmproj-Underdog-Saluki-27B-1.0-F16.gguf"         # 視覺檔，約 928 MB

MODEL  = hf_hub_download(REPO, MAIN,   local_dir="/kaggle/working/models")
MMPROJ = hf_hub_download(REPO, VISION, local_dir="/kaggle/working/models")
print(MODEL, MMPROJ, sep="\n")

## Cell 4：啟動伺服器

教材 F.1 的三個關鍵設定都在這裡：`-sm tensor`（張量並行）、`--spec-type ngram-mod`（推測解碼）、`-c`（上下文大小）。

`start_new_session=True` 讓伺服器獨立於這個 Cell —— 少了它，按下任何 Cell 的中斷都會把伺服器一起關掉。

成功時印出「服務就緒」。顯存不足時改用 `start(ctx=65536)`。

In [ ]:
import os, glob, secrets, subprocess, time, requests

API_KEY = secrets.token_urlsafe(24)      # 服務要對外，所以一定要金鑰
PORT = 8000
LOG = "/kaggle/working/server.log"

srv = glob.glob("/kaggle/working/llama/**/llama-server", recursive=True)[0]
libdirs = sorted({os.path.dirname(p) for p in
                  glob.glob("/kaggle/working/llama/**/*.so*", recursive=True)})
env = dict(os.environ, LD_LIBRARY_PATH=":".join(
    libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
server = None

def wait_ready(proc, timeout=900):
    begin = time.time()
    while time.time() - begin < timeout:
        if proc.poll() is not None:
            raise RuntimeError(f"伺服器已退出，請看 {LOG}")
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200:
                print("服務就緒"); return
        except requests.RequestException:
            pass
        time.sleep(5)
    raise TimeoutError(f"逾時，請看 {LOG}")

def start(ctx=131072):
    global server
    if server is not None and server.poll() is None:
        server.terminate(); server.wait()
    server = subprocess.Popen([
        srv, "-m", MODEL, "--mmproj", MMPROJ,
        "--alias", "saluki-27b",
        "--jinja", "-ngl", "99", "-fa", "on",
        "-sm", "tensor", "-ts", "1,1",
        "--spec-type", "ngram-mod",
        "-c", str(ctx), "-np", "1",
        "--host", "127.0.0.1", "--port", str(PORT), "--api-key", API_KEY,
    ], stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env,
       start_new_session=True)
    wait_ready(server)

start()
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

## Cell 5：驗收

五項全部印「通過」才往下做。另外**請親自讀一遍第 ② 題的完整回答**——自動檢查只看關鍵字有沒有出現，看得懂與否只有你能判斷。

參考值（2026-10-10 實測）：寫新程式 15.0–16.2 tok/s、改既有檔案 71.2–72.3 tok/s。

In [ ]:
import base64, io, inspect, textwrap
from PIL import Image, ImageDraw, ImageFont

def chat(content, n=400):
    r = requests.post(
        f"http://127.0.0.1:{PORT}/v1/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": "saluki-27b", "max_tokens": n, "temperature": 0,
              "chat_template_kwargs": {"enable_thinking": False},
              "messages": [{"role": "user", "content": content}]},
        timeout=900).json()
    return r["choices"][0]["message"]["content"], r["timings"]["predicted_per_second"]

text, _ = chat("用一句話自我介紹", 200)

img = Image.new("RGB", (1000, 240), "white")
ImageDraw.Draw(img).text((20, 20),
    'Traceback (most recent call last):\n  File "app.py", line 12, in <module>\n'
    '    total = price * qty\nTypeError: can\'t multiply sequence by non-int of type \'str\'',
    fill="black", font=ImageFont.load_default(size=24))
buf = io.BytesIO(); img.save(buf, format="PNG")
url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
seen, _ = chat([{"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": "這張截圖的錯誤是什麼？發生在哪個檔案的第幾行？怎麼修？"}], 800)

_, new_tps = chat("用 Python 寫一個完整的 LRU cache 類別並附上測試", 600)

SRC = inspect.getsource(textwrap)
_, edit_tps = chat(f"```python\n{SRC}\n```\n把 TextWrapper.wrap 方法開頭加上一行 "
                   "logging.debug 記錄輸入長度，然後輸出修改後的完整檔案，不要省略任何部分。", 600)

mem = [int(x) for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split()]

found = [k for k in ("TypeError", "app.py", "12") if k in seen]
checks = [
    ("文字回應",                 bool(text.strip()),         f"{len(text)} 字"),
    ("看圖讀出錯誤、檔名與行號", len(found) == 3,            f"找到 {found}"),
    ("寫新程式 >= 12 tok/s",     new_tps >= 12,              f"{new_tps:.1f}"),
    ("改既有檔案 >= 40 tok/s",   edit_tps >= 40,             f"{edit_tps:.1f}"),
    ("兩張卡都載入 > 6 GB",      all(m > 6000 for m in mem), f"{mem} MiB"),
]
for name, ok, detail in checks:
    print("通過" if ok else "失敗", "|", name, "|", detail)

print("\n===== 第 2 題的完整回答（請自己讀一遍）=====\n" + seen)

## Cell 6：開隧道，拿到網址與金鑰

把印出的 **Base URL** 與 **API Key** 填進你的代理式編碼工具。**這兩個值每次開機都不同。**

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared && /kaggle/working/cloudflared --version

In [ ]:
import re

CF_LOG = "/kaggle/working/cloudflared.log"
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate",
     "--url", f"http://127.0.0.1:{PORT}"],
    stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True)

url = None
for _ in range(30):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(CF_LOG).read())
    if m:
        url = m.group(0); break
    time.sleep(2)

print("Base URL:", url + "/v1" if url else f"尚未取得，請看 {CF_LOG}")
print("API Key :", API_KEY)

## Cell 7：保活（最後才執行）

兩個值都是 `None` 代表伺服器與隧道都還活著。要跑別的 Cell 就先中斷這一格，跑完再重新執行——中斷它**不會**影響伺服器與隧道。

**用完請回 Notebook 頁面按 Stop Session**，只關分頁不會停止計時。

**提交到版控前請清除所有輸出**：Cell 6 的輸出含有網址與金鑰。

In [ ]:
while True:
    print(time.strftime("%H:%M"), "server", server.poll(), "tunnel", tunnel.poll(), flush=True)
    time.sleep(300)